# Prompt Evaluations - AWS Dataset Generator

This notebook sets up a simple flow to generate a prompt evaluation dataset focused on AWS tasks that require Python, JSON, or Regex.

## Environment Setup

Loads environment variables, initializes the Anthropic client, and defines the base model for requests.

In [8]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helper Functions

Defines utility functions to build message history and wrap model calls with optional parameters such as system, temperature, and stop sequences.

In [9]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Dataset Generator

Creates the dataset generation prompt, enforces JSON output, and converts the response into a Python structure with json.loads for direct notebook usage.

In [10]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

## Run and Inspect

Runs dataset generation and displays the final result for a quick review of the created examples.

In [11]:
dataset = generate_dataset()
dataset

[{'task': 'Write a regular expression to validate an AWS S3 bucket name. S3 bucket names must be between 3 and 63 characters long, contain only lowercase letters, numbers, hyphens, and periods, start and end with a letter or number, and cannot contain consecutive hyphens or periods.'},
 {'task': "Write a Python function that takes an AWS IAM policy JSON string as input and returns a list of all the actions (e.g., 's3:GetObject', 'ec2:DescribeInstances') granted in that policy."},
 {'task': 'Create a JSON object that represents a valid AWS CloudFormation template for creating a simple VPC with a single public subnet, including required metadata and properties.'}]